<a href="https://colab.research.google.com/github/shanmukhag389/DocQuery-RAG/blob/main/DocQuery_RAG_NoteBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DocQuery-RAG: Advanced RAG Pipeline

An advanced Retrieval-Augmented Generation system built with hybrid search (dense + sparse + RRF),
cross-encoder re-ranking, and LLM-based evaluation. Built and tested in Google Colab, using
Gemini API for embeddings/generation and Qdrant Cloud for vector storage.

In [1]:
!pip install google-genai qdrant-client pymupdf langchain-text-splitters rank-bm25 flashrank -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 85.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
gemini_key = userdata.get('GEMINI_API_KEY')
print("Key loaded, starts with:", gemini_key[:5])

Key loaded, starts with: AQ.Ab


In [3]:
from google.colab import userdata

qdrant_url = userdata.get('QDRANT_URL')
qdrant_key = userdata.get('QDRANT_API_KEY')

print("Qdrant URL:", qdrant_url)
print("Qdrant Key loaded, starts with:", qdrant_key[:15])

Qdrant URL: https://26f5bffe-007f-4635-8303-3269227b299e.eu-west-1-0.aws.cloud.qdrant.io
Qdrant Key loaded, starts with: eyJhbGciOiJIUzI


In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Connect to your Qdrant Cloud cluster using the URL + key from Secrets
client = QdrantClient(url=qdrant_url, api_key=qdrant_key)

# Define the collection name - this is like naming a table in a database
COLLECTION_NAME = "docquery_chunks"

# Create the collection
# size=768 because Gemini's text-embedding-004 model outputs vectors of 768 numbers
# distance=COSINE means we measure "similarity" between vectors using cosine similarity,
# the standard choice for text embeddings
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

print("Collection created successfully!")

Collection created successfully!


In [9]:
import fitz

pdf_path = "/content/2607.24663v1.pdf"

doc = fitz.open(pdf_path)
print(f"Number of pages: {len(doc)}")

first_page_text = doc[0].get_text()
print("--- First page text preview ---")
print(first_page_text[:1000])

Number of pages: 38
--- First page text preview ---
APS-RAG
A corrective agentic hybrid RAG and an operations-grounded
evaluation for a scientific facility
Rajat Sainju,1, a) Dariusz Jarosz,1 Hairong Shang,1 Michael Prince,1 Ryan M. Aydelott,1 Mathew J. Cherukara,1
Yine Sun,1 and Michael Borland1
Advanced Photon Source, Argonne National Laboratory, Lemont, Illinois 60439, USA
(Dated: 28 July 2026)
Scientific user facilities accumulate decades of operational knowledge that no single search index covers: electronic
logbooks, technical documents, internal wikis, operations chat messages, maintenance records, and live control-system
data. We present APS-RAG—Advanced Photon Source Retrieval Augmented Generation—a deployed platform that
makes the institutional knowledge at the Advanced Photon Source (APS) accessible to staff through natural-language
queries, along with an operations-grounded evaluation. The retrieval engine fuses dense, sparse, and knowledge-graph
(KG) channels with query-ty

In [10]:
# Extract text from every page, not just the first
full_text = ""
for page in doc:
    full_text += page.get_text()

print(f"Total characters extracted: {len(full_text)}")

Total characters extracted: 201499


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size: max characters per chunk (roughly 150-200 words)
# chunk_overlap: how many characters repeat between consecutive chunks,
#                so a sentence split across a chunk boundary isn't lost entirely
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_text(full_text)

print(f"Number of chunks created: {len(chunks)}")
print("--- First chunk preview ---")
print(chunks[0])
print("\n--- Second chunk preview ---")
print(chunks[1])

Number of chunks created: 239
--- First chunk preview ---
APS-RAG
A corrective agentic hybrid RAG and an operations-grounded
evaluation for a scientific facility
Rajat Sainju,1, a) Dariusz Jarosz,1 Hairong Shang,1 Michael Prince,1 Ryan M. Aydelott,1 Mathew J. Cherukara,1
Yine Sun,1 and Michael Borland1
Advanced Photon Source, Argonne National Laboratory, Lemont, Illinois 60439, USA
(Dated: 28 July 2026)
Scientific user facilities accumulate decades of operational knowledge that no single search index covers: electronic
logbooks, technical documents, internal wikis, operations chat messages, maintenance records, and live control-system
data. We present APS-RAG—Advanced Photon Source Retrieval Augmented Generation—a deployed platform that
makes the institutional knowledge at the Advanced Photon Source (APS) accessible to staff through natural-language
queries, along with an operations-grounded evaluation. The retrieval engine fuses dense, sparse, and knowledge-graph

--- Second chunk pre

In [14]:
import time
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=gemini_key)

def get_embeddings(texts, batch_size=20, delay_seconds=15):
    """
    Takes a list of text chunks, returns a list of embedding vectors.
    Adds a delay between batches to respect the free-tier rate limit
    (100 embedding requests/minute).
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        # Retry loop in case we still hit a rate limit occasionally
        while True:
            try:
                result = gemini_client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=batch,
                    config=types.EmbedContentConfig(
                        task_type="RETRIEVAL_DOCUMENT",
                        output_dimensionality=768,
                    ),
                )
                break  # success, exit retry loop
            except Exception as e:
                if "RESOURCE_EXHAUSTED" in str(e):
                    print("Rate limit hit, waiting 20s before retry...")
                    time.sleep(20)
                else:
                    raise  # some other real error, don't hide it

        batch_embeddings = [e.values for e in result.embeddings]
        all_embeddings.extend(batch_embeddings)
        print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} chunks")

        time.sleep(delay_seconds)  # pace ourselves to stay under the per-minute limit

    return all_embeddings

chunk_embeddings = get_embeddings(chunks)
print(f"\nTotal embeddings generated: {len(chunk_embeddings)}")
print(f"Each embedding has {len(chunk_embeddings[0])} numbers")

Rate limit hit, waiting 20s before retry...
Processed 20/239 chunks
Processed 40/239 chunks
Processed 60/239 chunks
Processed 80/239 chunks
Processed 100/239 chunks
Processed 120/239 chunks
Processed 140/239 chunks
Processed 160/239 chunks
Processed 180/239 chunks
Processed 200/239 chunks
Processed 220/239 chunks
Processed 239/239 chunks

Total embeddings generated: 239
Each embedding has 768 numbers


In [15]:
from qdrant_client.models import PointStruct
import uuid

# Build a list of "points" - each point = one chunk's ID, vector, and the original text
points = []
for idx, (chunk_text, embedding) in enumerate(zip(chunks, chunk_embeddings)):
    points.append(
        PointStruct(
            id=idx,  # simple integer ID for each chunk
            vector=embedding,
            payload={"text": chunk_text, "chunk_index": idx},  # payload = extra data stored alongside the vector
        )
    )

# Upload all points to Qdrant in one go
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print(f"Uploaded {len(points)} points to Qdrant collection '{COLLECTION_NAME}'")

# Sanity check: ask Qdrant how many points it actually has now
collection_info = client.get_collection(COLLECTION_NAME)
print(f"Qdrant confirms: {collection_info.points_count} points stored")

Uploaded 239 points to Qdrant collection 'docquery_chunks'
Qdrant confirms: 239 points stored


In [16]:
from rank_bm25 import BM25Okapi

# BM25 needs each chunk broken into individual words (tokens), not as one big string
tokenized_chunks = [chunk.lower().split() for chunk in chunks]

# Build the BM25 index over all chunks
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

# Quick sanity test: search for a very specific term from the paper
test_query = "APS-Bench"
tokenized_query = test_query.lower().split()
scores = bm25.get_scores(tokenized_query)

# Get the index of the highest-scoring chunk
best_idx = scores.argmax()
print(f"\nTop BM25 match for '{test_query}':")
print(chunks[best_idx][:300])

BM25 index built over 239 chunks

Top BM25 match for 'APS-Bench':
The graph channel and corrective loop contribute positively as expected, but the performance gains are marginal. Ad-
ditionally, we also compare the performance of open-source and closed-source LLMs in final answer synthesis. We
release the APS-Bench construction methodology, the six-layer evaluatio


In [17]:
def dense_search(query, top_k=5):
    """
    Embeds the query, searches Qdrant for the most similar chunks by meaning.
    """
    query_embedding = gemini_client.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",  # different task_type than documents - optimized for short queries
            output_dimensionality=768,
        ),
    ).embeddings[0].values

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k,
    )
    return results.points

# Test with the same exact-match query as before
test_query = "APS-Bench"
dense_results = dense_search(test_query)

print(f"--- Dense (vector) search results for '{test_query}' ---")
for i, point in enumerate(dense_results):
    print(f"\n[{i+1}] Score: {point.score:.4f}")
    print(point.payload["text"][:200])


--- Dense (vector) search results for 'APS-Bench' ---

[1] Score: 0.7711
Evidence. APS_2170123__chunk_0 (primary)
APS-BENCH-005
factual · ICMS · easy
Q. According to DIAG-TN-2022-007, what ring is being considered as a
damping or accumulator ring to inject beam into the Li

[2] Score: 0.7521
running.
[o] checkDataLoggers helps identify which workstations need
attention when data archiving problems are suspected.
[o] checkDataLoggers helps identify which processes need attention when
data 

[3] Score: 0.7460
nC corresponds to a bunch current of 9.8 mA at 9.77 MHz.
[V] A bunch
charge of 21.9 nC corresponds to a bunch current of 214 mA at 9.77 MHz.
[V] The calibrated bunch-current range is 9.8 mA to 214 mA.

[4] Score: 0.7372
Evidence. 50694__chunk_0 (primary), 50362__chunk_0 (supporting)
APS-BENCH-024
comparative · ICMS · medium
Q. Compared with normal operation without VESD, what happened to the
PAR THz beamline CSR sign

[5] Score: 0.7372
[o] The cited passages identify the two tech no

In [18]:
def bm25_search(query, top_k=5):
    """
    Searches the BM25 index, returns top matching chunk indices with scores.
    """
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[::-1][:top_k]  # sort descending, take top_k
    return [(idx, scores[idx]) for idx in top_indices]


def hybrid_search_rrf(query, top_k=5, k=60):
    """
    Combines dense (Qdrant) and sparse (BM25) search results using
    Reciprocal Rank Fusion.

    RRF formula: score(doc) = sum over each ranked list of  1 / (k + rank)
    - rank is the doc's position in that list (1st place, 2nd place, etc.)
    - k=60 is a standard constant from the original RRF paper - it dampens
      the impact of very high individual ranks so no single method dominates
    """
    # Get more candidates from each method than we need, so fusion has room to work
    dense_results = dense_search(query, top_k=20)
    sparse_results = bm25_search(query, top_k=20)

    rrf_scores = {}

    # Add dense search's contribution
    for rank, point in enumerate(dense_results):
        chunk_idx = point.payload["chunk_index"]
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank + 1)

    # Add BM25's contribution
    for rank, (chunk_idx, _) in enumerate(sparse_results):
        rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0) + 1 / (k + rank + 1)

    # Sort all chunks by their combined RRF score, descending
    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    return sorted_chunks[:top_k]


# Test it with the same query
test_query = "APS-Bench"
fused_results = hybrid_search_rrf(test_query)

print(f"--- Hybrid RRF search results for '{test_query}' ---")
for i, (chunk_idx, score) in enumerate(fused_results):
    print(f"\n[{i+1}] RRF Score: {score:.5f}")
    print(chunks[chunk_idx][:200])

--- Hybrid RRF search results for 'APS-Bench' ---

[1] RRF Score: 0.01639
Evidence. APS_2170123__chunk_0 (primary)
APS-BENCH-005
factual · ICMS · easy
Q. According to DIAG-TN-2022-007, what ring is being considered as a
damping or accumulator ring to inject beam into the Li

[2] RRF Score: 0.01639
The graph channel and corrective loop contribute positively as expected, but the performance gains are marginal. Ad-
ditionally, we also compare the performance of open-source and closed-source LLMs i

[3] RRF Score: 0.01613
running.
[o] checkDataLoggers helps identify which workstations need
attention when data archiving problems are suspected.
[o] checkDataLoggers helps identify which processes need attention when
data 

[4] RRF Score: 0.01613
"type": "object",
"additionalProperties": false,
"required": ["entry_text", "attachment_urls"],
"properties": {
"entry_text":
{"type": "string"},
"attachment_urls": {"type": "array", "items": {"type":

[5] RRF Score: 0.01587
nC corresponds to a bunch 

In [19]:
from flashrank import Ranker, RerankRequest

# Initialize FlashRank - this downloads a small model file the first time (a few dozen MB)
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

def rerank_results(query, candidate_chunk_indices, top_k=5):
    """
    Takes candidate chunks (already narrowed down by RRF), re-scores them
    with a cross-encoder for real relevance, returns the cleaned-up top_k.
    """
    passages = [
        {"id": idx, "text": chunks[idx]}
        for idx in candidate_chunk_indices
    ]

    rerank_request = RerankRequest(query=query, passages=passages)
    reranked = ranker.rerank(rerank_request)

    return reranked[:top_k]


# Take the fused RRF candidates from before, feed them into the reranker
test_query = "APS-Bench"
candidate_indices = [chunk_idx for chunk_idx, _ in fused_results]

reranked_results = rerank_results(test_query, candidate_indices)

print(f"--- Re-ranked results for '{test_query}' ---")
for i, result in enumerate(reranked_results):
    print(f"\n[{i+1}] Rerank Score: {result['score']:.4f}")
    print(result['text'][:200])

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 51.9MiB/s]


--- Re-ranked results for 'APS-Bench' ---

[1] Rerank Score: 0.9966
The graph channel and corrective loop contribute positively as expected, but the performance gains are marginal. Ad-
ditionally, we also compare the performance of open-source and closed-source LLMs i

[2] Rerank Score: 0.9781
running.
[o] checkDataLoggers helps identify which workstations need
attention when data archiving problems are suspected.
[o] checkDataLoggers helps identify which processes need attention when
data 

[3] Rerank Score: 0.9529
Evidence. APS_2170123__chunk_0 (primary)
APS-BENCH-005
factual · ICMS · easy
Q. According to DIAG-TN-2022-007, what ring is being considered as a
damping or accumulator ring to inject beam into the Li

[4] Rerank Score: 0.8292
nC corresponds to a bunch current of 9.8 mA at 9.77 MHz.
[V] A bunch
charge of 21.9 nC corresponds to a bunch current of 214 mA at 9.77 MHz.
[V] The calibrated bunch-current range is 9.8 mA to 214 mA.

[5] Rerank Score: 0.0240
"type": "object",
"addit

In [21]:
def generate_answer(query, reranked_chunks, top_n=4):
    context = "\n\n---\n\n".join([r['text'] for r in reranked_chunks[:top_n]])

    prompt = f"""You are answering questions based ONLY on the provided context below.
If the answer is not present in the context, say "I cannot find this in the document" - do not guess or use outside knowledge.

Context:
{context}

Question: {query}

Answer:"""

    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    return response.text


test_query = "What is APS-Bench and how many questions does it contain?"

fused = hybrid_search_rrf(test_query, top_k=15)
candidate_idx = [chunk_idx for chunk_idx, _ in fused]
reranked = rerank_results(test_query, candidate_idx, top_k=5)

answer = generate_answer(test_query, reranked)
print("--- Generated Answer ---")
print(answer)

--- Generated Answer ---
Based on the provided context, **APS-Bench** is a corpus-derived benchmark built from the live corpus (rather than through manual annotation or handwriting) where question–answer pairs are generated from sampled corpus passages following the InPars methodology. 

It contains **50 questions** in total (49 answerable questions and one abstention item).


In [22]:
def evaluate_faithfulness(answer, context):
    """
    Asks Gemini to judge whether the answer is fully supported by the context,
    on a scale of 0-1. This is the LLM-as-judge pattern.
    """
    prompt = f"""You are a strict evaluator. Given a CONTEXT and an ANSWER, determine if every claim in the ANSWER is directly supported by the CONTEXT.

Context:
{context}

Answer:
{answer}

Respond with ONLY a number between 0 and 1:
- 1.0 = every claim in the answer is fully supported by the context
- 0.5 = some claims are supported, some are not
- 0.0 = the answer contains claims not found in the context at all

Respond with just the number, nothing else."""

    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    try:
        return float(response.text.strip())
    except ValueError:
        return None  # in case the model doesn't return a clean number


def evaluate_context_relevance(query, context):
    """
    Asks Gemini to judge whether the retrieved context is actually relevant
    to the question asked.
    """
    prompt = f"""You are a strict evaluator. Given a QUESTION and a CONTEXT, determine how relevant the context is to answering the question.

Question: {query}

Context:
{context}

Respond with ONLY a number between 0 and 1:
- 1.0 = the context is highly relevant and directly useful for answering
- 0.5 = the context is partially relevant
- 0.0 = the context is irrelevant to the question

Respond with just the number, nothing else."""

    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    try:
        return float(response.text.strip())
    except ValueError:
        return None


# Evaluate our previous answer
context_text = "\n\n---\n\n".join([r['text'] for r in reranked[:4]])

faithfulness_score = evaluate_faithfulness(answer, context_text)
relevance_score = evaluate_context_relevance(test_query, context_text)

print(f"Query: {test_query}")
print(f"Faithfulness Score: {faithfulness_score}")
print(f"Context Relevance Score: {relevance_score}")

Query: What is APS-Bench and how many questions does it contain?
Faithfulness Score: 1.0
Context Relevance Score: 1.0


In [25]:
import time

def safe_generate_content(prompt, model="gemini-3.6-flash", max_retries=5):
    """
    Wraps any Gemini generate_content call with automatic retry on rate limits.
    """
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )
            return response
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e):
                wait_time = 15
                print(f"Rate limit hit, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise
    raise Exception("Max retries exceeded")


# Update generate_answer to use the safe wrapper
def generate_answer(query, reranked_chunks, top_n=4):
    context = "\n\n---\n\n".join([r['text'] for r in reranked_chunks[:top_n]])
    prompt = f"""You are answering questions based ONLY on the provided context below.
If the answer is not present in the context, say "I cannot find this in the document" - do not guess or use outside knowledge.

Context:
{context}

Question: {query}

Answer:"""
    response = safe_generate_content(prompt)
    return response.text


# Update evaluate_faithfulness to use the safe wrapper
def evaluate_faithfulness(answer, context):
    prompt = f"""You are a strict evaluator. Given a CONTEXT and an ANSWER, determine if every claim in the ANSWER is directly supported by the CONTEXT.

Context:
{context}

Answer:
{answer}

Respond with ONLY a number between 0 and 1:
- 1.0 = every claim in the answer is fully supported by the context
- 0.5 = some claims are supported, some are not
- 0.0 = the answer contains claims not found in the context at all

Respond with just the number, nothing else."""
    response = safe_generate_content(prompt)
    try:
        return float(response.text.strip())
    except ValueError:
        return None


# Update evaluate_context_relevance to use the safe wrapper
def evaluate_context_relevance(query, context):
    prompt = f"""You are a strict evaluator. Given a QUESTION and a CONTEXT, determine how relevant the context is to answering the question.

Question: {query}

Context:
{context}

Respond with ONLY a number between 0 and 1:
- 1.0 = the context is highly relevant and directly useful for answering
- 0.5 = the context is partially relevant
- 0.0 = the context is irrelevant to the question

Respond with just the number, nothing else."""
    response = safe_generate_content(prompt)
    try:
        return float(response.text.strip())
    except ValueError:
        return None

In [26]:
test_query = "What is the capital of France?"
result = run_full_pipeline(test_query)
print(f"Q: {result['query']}")
print(f"A: {result['answer']}")
print(f"Faithfulness: {result['faithfulness']} | Relevance: {result['relevance']}")

Q: What is the capital of France?
A: I cannot find this in the document
Faithfulness: 1.0 | Relevance: 0.0
